# RappiPay: Fraud Detection Pipeline + React App con Cortex Code

## Laboratorio Tecnico — Data Engineers & Analysts

**Objetivo**: Construir un pipeline de deteccion de fraude end-to-end en Snowflake y crear un Dashboard React con Cortex Code para investigar alertas.

**Audiencia**: Data Engineers y Analysts (nivel intermedio)

**Duracion estimada**: 45-60 minutos

**Productos Snowflake utilizados**:
- **Dynamic Tables** — Pipeline declarativo Bronze > Silver > Gold
- **Cortex AI Functions** — AI_CLASSIFY, AI_EXTRACT, AI_SENTIMENT, COMPLETE
- **Semantic Views** — Cortex Analyst para consultas en lenguaje natural
- **Cortex Code** — Generacion de React App (Snowflake App Runtime)

**Nota**: Los datos utilizados son ficticios pero inspirados en informacion real del mercado fintech colombiano/mexicano.

---

### Arquitectura del Lab

```
Transacciones (raw) --> Dynamic Tables (Bronze>Silver>Gold) --> Cortex AI (enriquecimiento)
                                                                        |
                                                                        v
                                                              Semantic View (Analyst)
                                                                        |
                                                                        v
                                                        React App (Cortex Code) --> Usuarios
```

## Prerequisitos

Antes de comenzar, asegurate de tener:

| Requisito | Detalle |
|-----------|----------|
| Snowflake Account | Con ACCOUNTADMIN o un rol con permisos de CREATE DATABASE, CREATE WAREHOUSE |
| Cortex Code | CLI instalado (`npm i -g @snowflake-labs/cortex-code`) o Desktop App |
| Node.js 18+ | Requerido para la React App (solo si ejecutas localmente) |
| Cortex AI Functions | Habilitadas en tu cuenta (disponibles en la mayoria de regiones) |

> **Tip**: Si no tienes Cortex Code instalado, puedes usar Snowflake CLI (`snow`) con el comando `snow app` para desplegar la aplicacion.

## Seccion 1: Setup del Ambiente

Creamos la base de datos, esquemas, warehouse y roles necesarios para el laboratorio.

> Para el setup completo con datos extendidos, ejecuta `scripts/one_click_run.sql`

In [ ]:
-- Snowflake SQL
-- =============================================================================
-- SECCION 1: SETUP DEL AMBIENTE
-- =============================================================================

USE ROLE ACCOUNTADMIN;

-- Crear warehouse dedicado
CREATE WAREHOUSE IF NOT EXISTS RAPPIPAY_WH
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 120
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

-- Crear base de datos con esquemas por capa
CREATE DATABASE IF NOT EXISTS RAPPIPAY_DB;

CREATE SCHEMA IF NOT EXISTS RAPPIPAY_DB.RAW;         -- Bronze: datos crudos
CREATE SCHEMA IF NOT EXISTS RAPPIPAY_DB.CURATED;     -- Silver: datos limpios y enriquecidos
CREATE SCHEMA IF NOT EXISTS RAPPIPAY_DB.ANALYTICS;   -- Gold: metricas y agregaciones

USE WAREHOUSE RAPPIPAY_WH;
USE DATABASE RAPPIPAY_DB;
USE SCHEMA RAW;

## Seccion 2: Datos de Ejemplo

Insertamos datos ficticios que simulan transacciones de una fintech: pagos QR, transferencias P2P, recargas, y compras en comercios.

In [ ]:
-- Snowflake SQL
-- =============================================================================
-- SECCION 2: TABLAS BASE Y DATOS DE EJEMPLO
-- =============================================================================

-- Tabla de clientes
CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.CUSTOMERS (
    customer_id VARCHAR(36) DEFAULT UUID_STRING(),
    full_name VARCHAR(200),
    email VARCHAR(200),
    phone VARCHAR(20),
    country VARCHAR(3),
    city VARCHAR(100),
    account_tier VARCHAR(20),       -- BASICO, PLUS, PREMIUM
    kyc_level INT,                  -- 1=basico, 2=validado, 3=completo
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Tabla de comercios
CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.MERCHANTS (
    merchant_id VARCHAR(36) DEFAULT UUID_STRING(),
    merchant_name VARCHAR(200),
    category VARCHAR(50),           -- RESTAURANTE, SUPERMERCADO, ELECTRONICA, etc.
    mcc_code VARCHAR(4),
    city VARCHAR(100),
    country VARCHAR(3),
    risk_score FLOAT DEFAULT 0.0
);

-- Tabla de transacciones (fuente principal)
CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.TRANSACTIONS (
    transaction_id VARCHAR(36) DEFAULT UUID_STRING(),
    customer_id VARCHAR(36),
    merchant_id VARCHAR(36),
    amount DECIMAL(18,2),
    currency VARCHAR(3),            -- COP, MXN, BRL
    transaction_type VARCHAR(30),   -- QR_PAYMENT, P2P_TRANSFER, RECHARGE, CARD_PURCHASE
    channel VARCHAR(20),            -- APP, WEB, POS
    device_id VARCHAR(100),
    ip_address VARCHAR(45),
    geolocation VARCHAR(50),
    status VARCHAR(20),             -- COMPLETED, PENDING, DECLINED, FLAGGED
    description VARCHAR(500),
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Tabla de alertas de fraude
CREATE OR REPLACE TABLE RAPPIPAY_DB.RAW.FRAUD_ALERTS (
    alert_id VARCHAR(36) DEFAULT UUID_STRING(),
    transaction_id VARCHAR(36),
    customer_id VARCHAR(36),
    alert_type VARCHAR(50),         -- VELOCITY, AMOUNT_ANOMALY, GEO_MISMATCH, DEVICE_NEW
    severity VARCHAR(10),           -- LOW, MEDIUM, HIGH, CRITICAL
    risk_score FLOAT,
    investigator_notes VARCHAR(2000),
    customer_communication VARCHAR(2000),
    status VARCHAR(20),             -- OPEN, INVESTIGATING, RESOLVED, ESCALATED
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Insertar clientes de ejemplo
INSERT INTO RAPPIPAY_DB.RAW.CUSTOMERS (customer_id, full_name, email, phone, country, city, account_tier, kyc_level)
VALUES
    ('C001', 'Maria Fernanda Lopez', 'mflopez@email.com', '+573001234567', 'COL', 'Bogota', 'PREMIUM', 3),
    ('C002', 'Carlos Andres Ramirez', 'caramirez@email.com', '+573109876543', 'COL', 'Medellin', 'PLUS', 2),
    ('C003', 'Ana Sofia Martinez', 'asmartinez@email.com', '+525512345678', 'MEX', 'CDMX', 'BASICO', 1),
    ('C004', 'Juan Pablo Hernandez', 'jphernandez@email.com', '+525598765432', 'MEX', 'Guadalajara', 'PREMIUM', 3),
    ('C005', 'Valentina Garcia Ruiz', 'vgarcia@email.com', '+573205551234', 'COL', 'Cali', 'PLUS', 2),
    ('C006', 'Diego Alejandro Torres', 'datorres@email.com', '+573114447890', 'COL', 'Barranquilla', 'BASICO', 1),
    ('C007', 'Camila Andrea Restrepo', 'carestrepo@email.com', '+573008889012', 'COL', 'Bogota', 'PREMIUM', 3),
    ('C008', 'Roberto Carlos Mendez', 'rcmendez@email.com', '+525533334444', 'MEX', 'Monterrey', 'PLUS', 2);

-- Insertar comercios
INSERT INTO RAPPIPAY_DB.RAW.MERCHANTS (merchant_id, merchant_name, category, mcc_code, city, country, risk_score)
VALUES
    ('M001', 'Restaurante El Cielo', 'RESTAURANTE', '5812', 'Bogota', 'COL', 0.1),
    ('M002', 'Exito Supermercados', 'SUPERMERCADO', '5411', 'Medellin', 'COL', 0.05),
    ('M003', 'Elektra Mexico', 'ELECTRONICA', '5732', 'CDMX', 'MEX', 0.3),
    ('M004', 'Falabella Online', 'ECOMMERCE', '5999', 'Bogota', 'COL', 0.2),
    ('M005', 'Casino Virtual LatAm', 'GAMBLING', '7995', 'Panama', 'PAN', 0.85),
    ('M006', 'Crypto Exchange XYZ', 'CRYPTO', '6051', 'CDMX', 'MEX', 0.9),
    ('M007', 'Farmacia Colsubsidio', 'FARMACIA', '5912', 'Bogota', 'COL', 0.05),
    ('M008', 'Rappi Travel', 'VIAJES', '4722', 'Bogota', 'COL', 0.15);

-- Insertar transacciones (mezcla de normales y sospechosas)
INSERT INTO RAPPIPAY_DB.RAW.TRANSACTIONS 
    (transaction_id, customer_id, merchant_id, amount, currency, transaction_type, channel, device_id, ip_address, geolocation, status, description, created_at)
VALUES
    ('T001', 'C001', 'M001', 85000, 'COP', 'QR_PAYMENT', 'APP', 'dev-001', '181.49.100.1', '4.7110,-74.0721', 'COMPLETED', 'Almuerzo ejecutivo para 2 personas', DATEADD(hour, -2, CURRENT_TIMESTAMP())),
    ('T002', 'C001', 'M002', 250000, 'COP', 'CARD_PURCHASE', 'POS', 'dev-001', '181.49.100.1', '6.2442,-75.5812', 'COMPLETED', 'Mercado semanal supermercado', DATEADD(hour, -1, CURRENT_TIMESTAMP())),
    ('T003', 'C002', 'M005', 5000000, 'COP', 'CARD_PURCHASE', 'WEB', 'dev-unknown', '45.33.32.156', '9.0,-79.5', 'FLAGGED', 'Deposito casino online - monto inusual desde IP extranjera', DATEADD(minute, -45, CURRENT_TIMESTAMP())),
    ('T004', 'C003', 'M006', 150000, 'MXN', 'P2P_TRANSFER', 'APP', 'dev-003', '189.203.100.5', '19.4326,-99.1332', 'FLAGGED', 'Compra crypto USDT - patron de fraccionamiento detectado', DATEADD(minute, -30, CURRENT_TIMESTAMP())),
    ('T005', 'C004', 'M003', 45000, 'MXN', 'CARD_PURCHASE', 'APP', 'dev-004', '189.203.55.10', '20.6597,-103.3496', 'COMPLETED', 'Compra audifonos inalambricos', DATEADD(hour, -3, CURRENT_TIMESTAMP())),
    ('T006', 'C002', 'M005', 3000000, 'COP', 'CARD_PURCHASE', 'WEB', 'dev-unknown', '45.33.32.156', '9.0,-79.5', 'FLAGGED', 'Segundo deposito casino - mismo dia dispositivo no reconocido', DATEADD(minute, -20, CURRENT_TIMESTAMP())),
    ('T007', 'C005', 'M004', 1200000, 'COP', 'CARD_PURCHASE', 'APP', 'dev-005', '181.49.55.20', '3.4516,-76.5320', 'COMPLETED', 'Compra zapatos y ropa online', DATEADD(hour, -5, CURRENT_TIMESTAMP())),
    ('T008', 'C006', 'M006', 8000000, 'COP', 'P2P_TRANSFER', 'WEB', 'dev-new-999', '103.21.44.8', '35.6762,139.6503', 'FLAGGED', 'Transferencia a exchange crypto - IP desde Tokio usuario en Barranquilla', DATEADD(minute, -15, CURRENT_TIMESTAMP())),
    ('T009', 'C007', 'M007', 45000, 'COP', 'QR_PAYMENT', 'APP', 'dev-007', '181.49.88.30', '4.7110,-74.0721', 'COMPLETED', 'Compra medicamentos recetados', DATEADD(hour, -4, CURRENT_TIMESTAMP())),
    ('T010', 'C008', 'M008', 3500000, 'MXN', 'CARD_PURCHASE', 'APP', 'dev-008', '189.203.77.15', '25.6866,-100.3161', 'COMPLETED', 'Vuelo Monterrey-Cancun ida y vuelta', DATEADD(hour, -6, CURRENT_TIMESTAMP())),
    ('T011', 'C003', 'M006', 200000, 'MXN', 'P2P_TRANSFER', 'APP', 'dev-003', '189.203.100.5', '19.4326,-99.1332', 'FLAGGED', 'Tercera compra crypto en 1 hora - patron smurfing', DATEADD(minute, -10, CURRENT_TIMESTAMP())),
    ('T012', 'C001', 'M004', 890000, 'COP', 'CARD_PURCHASE', 'WEB', 'dev-new-777', '91.108.56.1', '55.7558,37.6173', 'FLAGGED', 'Compra electronica premium - IP desde Moscu usuario en Bogota', DATEADD(minute, -5, CURRENT_TIMESTAMP()));

-- Insertar alertas de fraude
INSERT INTO RAPPIPAY_DB.RAW.FRAUD_ALERTS
    (alert_id, transaction_id, customer_id, alert_type, severity, risk_score, investigator_notes, customer_communication, status, created_at)
VALUES
    ('A001', 'T003', 'C002', 'AMOUNT_ANOMALY', 'HIGH', 0.87, 
     'Cliente con patron de gasto promedio de 300K COP realiza transaccion de 5M en casino online. IP no coincide con ubicacion habitual (Panama vs Medellin). Dispositivo no registrado previamente.',
     'Hola Carlos, detectamos una transaccion inusual en tu cuenta por $5.000.000 en un casino online. Por favor confirma si fuiste tu. Estamos preocupados por la seguridad de tu cuenta.',
     'INVESTIGATING', DATEADD(minute, -44, CURRENT_TIMESTAMP())),
    ('A002', 'T004', 'C003', 'VELOCITY', 'MEDIUM', 0.72,
     'Multiples transferencias a exchange crypto en intervalo corto. Patron consistente con fraccionamiento (smurfing). KYC nivel 1 - verificacion incompleta.',
     'Ana, notamos varias transferencias rapidas desde tu cuenta. Para tu seguridad, necesitamos verificar tu identidad. Puedes completar la verificacion en la app.',
     'OPEN', DATEADD(minute, -29, CURRENT_TIMESTAMP())),
    ('A003', 'T006', 'C002', 'VELOCITY', 'CRITICAL', 0.94,
     'Segunda transaccion de alto monto al mismo casino en menos de 1 hora. Dispositivo desconocido. IP extranjera. Posible account takeover o abuso de credenciales robadas. URGENTE: bloquear cuenta.',
     'Carlos, hemos bloqueado temporalmente tu cuenta por actividad sospechosa. Dos transacciones de alto valor a casino online en 25 minutos. Contactanos al 018000-123-456 inmediatamente.',
     'ESCALATED', DATEADD(minute, -19, CURRENT_TIMESTAMP())),
    ('A004', 'T008', 'C006', 'GEO_MISMATCH', 'CRITICAL', 0.96,
     'Transferencia de 8M COP a crypto exchange desde IP en Tokio. Usuario registrado en Barranquilla, Colombia. Dispositivo nuevo nunca visto. Hora inusual (3am hora local). Todos los indicadores apuntan a account takeover.',
     'Diego, detectamos un acceso a tu cuenta desde Japon que no coincide con tu ubicacion. Hemos bloqueado la transferencia. Si no fuiste tu, cambia tu clave inmediatamente.',
     'ESCALATED', DATEADD(minute, -14, CURRENT_TIMESTAMP())),
    ('A005', 'T011', 'C003', 'VELOCITY', 'HIGH', 0.83,
     'Tercera operacion crypto del mismo cliente en 1 hora. Patron clasico de smurfing - multiples transacciones por debajo del umbral de reporte. Total acumulado supera limite de alerta.',
     'Ana, por regulacion debemos verificar las transferencias frecuentes a exchanges. Tu cuenta queda en revision hasta completar verificacion KYC nivel 2.',
     'INVESTIGATING', DATEADD(minute, -9, CURRENT_TIMESTAMP())),
    ('A006', 'T012', 'C001', 'GEO_MISMATCH', 'HIGH', 0.81,
     'Compra online desde IP en Moscu. Cliente premium con historial limpio pero dispositivo nuevo y geolocalizacion anomala. Posible phishing o compromiso de credenciales.',
     'Maria Fernanda, una compra de $890.000 se realizo desde una ubicacion inusual. Si no la reconoces, reportala en la app y bloquearemos tu tarjeta.',
     'OPEN', DATEADD(minute, -4, CURRENT_TIMESTAMP()));

## Seccion 3: Pipeline con Dynamic Tables

Construimos un pipeline declarativo **Bronze > Silver > Gold** usando Dynamic Tables.

**Ventajas de Dynamic Tables**:
- No requieren Streams + Tasks manuales
- Snowflake maneja automaticamente el refresh y las dependencias
- `TARGET_LAG` define la frescura maxima aceptable de los datos
- `DOWNSTREAM` optimiza refreshes intermedios (solo se actualizan cuando se necesitan downstream)

In [ ]:
-- Snowflake SQL
-- =============================================================================
-- SECCION 3: DYNAMIC TABLES — PIPELINE BRONZE > SILVER > GOLD
-- =============================================================================

-- SILVER: Transacciones enriquecidas con datos de cliente y comercio
CREATE OR REPLACE DYNAMIC TABLE RAPPIPAY_DB.CURATED.TRANSACTIONS_ENRICHED
    TARGET_LAG = '1 minute'
    WAREHOUSE = RAPPIPAY_WH
AS
SELECT
    t.transaction_id,
    t.customer_id,
    c.full_name AS customer_name,
    c.country AS customer_country,
    c.city AS customer_city,
    c.account_tier,
    c.kyc_level,
    t.merchant_id,
    m.merchant_name,
    m.category AS merchant_category,
    m.risk_score AS merchant_risk_score,
    t.amount,
    t.currency,
    t.transaction_type,
    t.channel,
    t.device_id,
    t.ip_address,
    t.geolocation,
    t.status,
    t.description,
    t.created_at,
    -- Indicadores de riesgo calculados
    CASE 
        WHEN m.risk_score > 0.7 THEN 'HIGH_RISK_MERCHANT'
        WHEN t.amount > 2000000 AND t.currency = 'COP' THEN 'HIGH_AMOUNT'
        WHEN t.amount > 100000 AND t.currency = 'MXN' THEN 'HIGH_AMOUNT'
        WHEN t.status = 'FLAGGED' THEN 'SYSTEM_FLAGGED'
        ELSE 'NORMAL'
    END AS risk_indicator,
    -- Flag si la IP no es de LATAM (simplificado)
    CASE
        WHEN SPLIT_PART(t.ip_address, '.', 1)::INT NOT BETWEEN 170 AND 200 
             AND t.status = 'FLAGGED' THEN TRUE
        ELSE FALSE
    END AS foreign_ip_flag
FROM RAPPIPAY_DB.RAW.TRANSACTIONS t
LEFT JOIN RAPPIPAY_DB.RAW.CUSTOMERS c ON t.customer_id = c.customer_id
LEFT JOIN RAPPIPAY_DB.RAW.MERCHANTS m ON t.merchant_id = m.merchant_id;


-- GOLD: Metricas de fraude por hora
CREATE OR REPLACE DYNAMIC TABLE RAPPIPAY_DB.ANALYTICS.FRAUD_METRICS_HOURLY
    TARGET_LAG = '5 minutes'
    WAREHOUSE = RAPPIPAY_WH
AS
SELECT
    DATE_TRUNC('hour', te.created_at) AS metric_hour,
    te.customer_country,
    te.merchant_category,
    te.transaction_type,
    COUNT(*) AS total_transactions,
    COUNT(CASE WHEN te.status = 'FLAGGED' THEN 1 END) AS flagged_transactions,
    ROUND(COUNT(CASE WHEN te.status = 'FLAGGED' THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0), 2) AS fraud_rate_pct,
    SUM(te.amount) AS total_amount,
    SUM(CASE WHEN te.status = 'FLAGGED' THEN te.amount ELSE 0 END) AS flagged_amount,
    AVG(CASE WHEN te.status = 'FLAGGED' THEN te.amount END) AS avg_flagged_amount,
    COUNT(DISTINCT te.customer_id) AS unique_customers,
    COUNT(DISTINCT CASE WHEN te.status = 'FLAGGED' THEN te.customer_id END) AS unique_flagged_customers,
    COUNT(CASE WHEN te.foreign_ip_flag THEN 1 END) AS foreign_ip_count
FROM RAPPIPAY_DB.CURATED.TRANSACTIONS_ENRICHED te
GROUP BY 1, 2, 3, 4;


-- Verificar que las Dynamic Tables se crearon correctamente
SHOW DYNAMIC TABLES IN SCHEMA RAPPIPAY_DB.CURATED;
SHOW DYNAMIC TABLES IN SCHEMA RAPPIPAY_DB.ANALYTICS;

## Seccion 4: Enriquecimiento con Cortex AI

Usamos funciones de Cortex AI para analizar texto no estructurado en las alertas de fraude:

| Funcion | Uso |
|---------|-----|
| `AI_CLASSIFY` | Clasificar descripciones de transacciones por tipo de fraude |
| `AI_EXTRACT` | Extraer entidades clave de notas de investigador |
| `AI_SENTIMENT` | Analizar tono de comunicaciones con clientes |
| `SNOWFLAKE.CORTEX.COMPLETE` | Generar resumenes ejecutivos de alertas |

In [ ]:
-- Snowflake SQL
-- =============================================================================
-- SECCION 4: CORTEX AI FUNCTIONS — ENRIQUECIMIENTO DE ALERTAS
-- =============================================================================

-- 4.1 AI_CLASSIFY: Clasificar transacciones por tipo de fraude
SELECT 
    t.transaction_id,
    t.description,
    AI_CLASSIFY(
        t.description,
        ['Lavado de dinero / Smurfing', 'Account Takeover', 'Fraude en comercio', 'Transaccion legitima', 'Phishing / Ingenieria social']
    ) AS fraud_classification
FROM RAPPIPAY_DB.RAW.TRANSACTIONS t
WHERE t.status = 'FLAGGED';

In [ ]:
-- Snowflake SQL
-- 4.2 AI_EXTRACT: Extraer entidades de notas del investigador
SELECT
    a.alert_id,
    a.alert_type,
    AI_EXTRACT(
        a.investigator_notes,
        ['monto_involucrado', 'ubicacion_sospechosa', 'tipo_de_ataque', 'accion_recomendada']
    ) AS extracted_entities
FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS a
WHERE a.status IN ('INVESTIGATING', 'ESCALATED');

In [ ]:
-- Snowflake SQL
-- 4.3 AI_SENTIMENT: Analizar comunicaciones con clientes
SELECT
    a.alert_id,
    a.customer_id,
    AI_SENTIMENT(a.customer_communication) AS sentiment_score,
    CASE
        WHEN AI_SENTIMENT(a.customer_communication) < -0.3 THEN 'NEGATIVO'
        WHEN AI_SENTIMENT(a.customer_communication) > 0.3 THEN 'POSITIVO'
        ELSE 'NEUTRAL'
    END AS sentiment_label
FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS a
WHERE a.customer_communication IS NOT NULL;

In [ ]:
-- Snowflake SQL
-- 4.4 SNOWFLAKE.CORTEX.COMPLETE: Resumen ejecutivo de alertas
SELECT
    a.alert_id,
    a.severity,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Genera un resumen ejecutivo de 2 lineas en espanol de esta alerta de fraude para el equipo de compliance. ',
            'Incluye: tipo de riesgo, monto, y accion sugerida. ',
            'Notas del investigador: ', a.investigator_notes
        )
    ) AS executive_summary
FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS a
WHERE a.severity IN ('HIGH', 'CRITICAL');

In [ ]:
-- Snowflake SQL
-- 4.5 Vista consolidada: Alertas enriquecidas con AI
CREATE OR REPLACE VIEW RAPPIPAY_DB.ANALYTICS.FRAUD_ALERTS_ENRICHED AS
SELECT
    a.alert_id,
    a.transaction_id,
    a.customer_id,
    c.full_name AS customer_name,
    c.account_tier,
    a.alert_type,
    a.severity,
    a.risk_score,
    a.status,
    a.created_at,
    t.amount,
    t.currency,
    t.merchant_id,
    m.merchant_name,
    m.category AS merchant_category,
    t.transaction_type,
    t.channel,
    -- Enriquecimiento AI
    AI_CLASSIFY(
        t.description,
        ['Lavado de dinero / Smurfing', 'Account Takeover', 'Fraude en comercio', 'Transaccion legitima', 'Phishing / Ingenieria social']
    ):label::VARCHAR AS ai_fraud_type,
    AI_SENTIMENT(a.customer_communication) AS communication_sentiment,
    a.investigator_notes,
    a.customer_communication
FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS a
JOIN RAPPIPAY_DB.RAW.TRANSACTIONS t ON a.transaction_id = t.transaction_id
JOIN RAPPIPAY_DB.RAW.CUSTOMERS c ON a.customer_id = c.customer_id
LEFT JOIN RAPPIPAY_DB.RAW.MERCHANTS m ON t.merchant_id = m.merchant_id;

## Seccion 5: Semantic Views para Cortex Analyst

Creamos una Semantic View que permite a Cortex Analyst responder preguntas sobre fraude en lenguaje natural (espanol, ingles, portugues).

La Semantic View define:
- **Tablas base** y sus relaciones
- **Metricas** calculadas (tasa de fraude, monto en riesgo)
- **Filtros** recomendados
- **Preguntas ejemplo** via VQRs (Verified Query Representations)

In [ ]:
-- Snowflake SQL
-- =============================================================================
-- SECCION 5: SEMANTIC VIEW PARA CORTEX ANALYST
-- =============================================================================

CREATE OR REPLACE SEMANTIC VIEW RAPPIPAY_DB.ANALYTICS.SV_FRAUD_ANALYTICS
  COMMENT = $$
  Vista semantica para analisis de fraude en RappiPay.
  Permite consultas en lenguaje natural sobre alertas, transacciones y metricas.
  
  Preguntas ejemplo:
  - Cuantas alertas criticas hay abiertas hoy?
  - Cual es la tasa de fraude por pais en las ultimas 24 horas?
  - Que comercios tienen mas transacciones flaggeadas?
  - Muestrame el monto total en riesgo por tipo de fraude
  - How many high severity alerts are escalated?
  - What is the average risk score by merchant category?
  - Qual a taxa de fraude por tipo de transacao?
  $$
  AS
  TABLES (
    RAPPIPAY_DB.ANALYTICS.FRAUD_ALERTS_ENRICHED
      AS fraud_alerts
      WITH SEMANTICS (
        alert_id IDENTIFIER 'ID unico de la alerta',
        customer_name DIMENSION 'Nombre completo del cliente',
        alert_type DIMENSION 'Tipo de alerta: VELOCITY, AMOUNT_ANOMALY, GEO_MISMATCH, DEVICE_NEW',
        severity DIMENSION 'Severidad: LOW, MEDIUM, HIGH, CRITICAL',
        status DIMENSION 'Estado: OPEN, INVESTIGATING, RESOLVED, ESCALATED',
        merchant_category DIMENSION 'Categoria del comercio',
        merchant_name DIMENSION 'Nombre del comercio',
        transaction_type DIMENSION 'Tipo: QR_PAYMENT, P2P_TRANSFER, RECHARGE, CARD_PURCHASE',
        currency DIMENSION 'Moneda: COP, MXN, BRL',
        ai_fraud_type DIMENSION 'Clasificacion AI del tipo de fraude',
        risk_score MEASURE 'Score de riesgo de 0 a 1',
        amount MEASURE 'Monto de la transaccion en moneda local',
        communication_sentiment MEASURE 'Sentimiento de la comunicacion (-1 a 1)',
        created_at DATETIME 'Fecha y hora de creacion de la alerta'
      ),
    RAPPIPAY_DB.ANALYTICS.FRAUD_METRICS_HOURLY
      AS fraud_metrics
      WITH SEMANTICS (
        metric_hour DATETIME 'Hora de la metrica',
        customer_country DIMENSION 'Pais del cliente: COL, MEX, BRL',
        merchant_category DIMENSION 'Categoria del comercio',
        transaction_type DIMENSION 'Tipo de transaccion',
        total_transactions MEASURE 'Total de transacciones en la hora',
        flagged_transactions MEASURE 'Transacciones marcadas como sospechosas',
        fraud_rate_pct MEASURE 'Porcentaje de fraude (flagged/total * 100)',
        total_amount MEASURE 'Monto total transaccionado',
        flagged_amount MEASURE 'Monto total en riesgo (transacciones flagged)',
        unique_customers MEASURE 'Clientes unicos',
        foreign_ip_count MEASURE 'Transacciones con IP extranjera'
      )
  );

-- Verificar la Semantic View
DESCRIBE SEMANTIC VIEW RAPPIPAY_DB.ANALYTICS.SV_FRAUD_ANALYTICS;

### Probar Cortex Analyst con la Semantic View

Una vez creada la Semantic View, puedes hacer preguntas en lenguaje natural desde:
- **Snowsight** > Cortex Analyst (chat)
- **Cortex Code** > `@analyst Cuantas alertas criticas hay?`
- **API REST** > `POST /api/v2/cortex/analyst/message`

Ejemplo de consulta directa:

In [ ]:
-- Snowflake SQL
-- Consulta equivalente a: "Cual es el monto total en riesgo por tipo de alerta?"
SELECT
    alert_type,
    severity,
    COUNT(*) AS num_alerts,
    SUM(amount) AS total_amount_at_risk,
    AVG(risk_score) AS avg_risk_score
FROM RAPPIPAY_DB.ANALYTICS.FRAUD_ALERTS_ENRICHED
GROUP BY alert_type, severity
ORDER BY total_amount_at_risk DESC;

## Seccion 6: Construir React App con Cortex Code

### Que es Cortex Code?

Cortex Code es el asistente de desarrollo AI de Snowflake que puede generar aplicaciones completas desde una descripcion en lenguaje natural. Las aplicaciones se despliegan directamente en Snowflake (App Runtime) sin necesidad de infraestructura externa.

### Instrucciones

**Paso 1**: Abre Cortex Code (CLI o Desktop)

```bash
# CLI
cortex

# O abre Cortex Code Desktop
```

**Paso 2**: Copia y pega el siguiente prompt (tambien disponible en `react_app_prompt.md`):

---

> **Prompt para Cortex Code:**
>
> Crea una aplicacion React para monitoreo de fraude de RappiPay que se conecte a RAPPIPAY_DB.
> La app debe incluir:
> - Header con branding RappiPay (color naranja #FF6B00) y status de conexion a Snowflake
> - 4 KPI cards: transacciones hoy, alertas activas, tasa de fraude (%), monto en riesgo (COP)
> - Grafico de linea con tendencia de alertas ultimas 24h (usar Recharts)
> - Tabla interactiva de alertas filtrable por severidad, tipo de fraude, merchant
> - Panel de detalle: al click en una alerta, mostrar transacciones asociadas y risk score
> - Filtros globales: rango de fecha, tipo de fraude, categoria merchant, monto minimo
> - Stack: React + Tailwind CSS + Recharts
> - Conectar a las tablas:
>   - RAPPIPAY_DB.ANALYTICS.FRAUD_METRICS_HOURLY
>   - RAPPIPAY_DB.ANALYTICS.FRAUD_ALERTS_ENRICHED
>   - RAPPIPAY_DB.CURATED.TRANSACTIONS_ENRICHED

---

**Paso 3**: Cortex Code generara:
- Estructura del proyecto (Next.js + Tailwind + Recharts)
- Componentes: `Dashboard`, `AlertTable`, `KPICard`, `TrendChart`, `AlertDetail`
- Conexion a Snowflake via `querySnowflake()` SDK helper
- Queries SQL embebidos para cada componente
- Estilos con paleta RappiPay

**Paso 4**: Revisa el codigo generado y despliega:

```bash
# Probar localmente
npm run dev

# Desplegar a Snowflake App Runtime
snow app deploy
```

**Paso 5**: Accede a la app desplegada desde Snowsight > Apps

### Resultado esperado

```
+------------------------------------------------------------------+
| RAPPIPAY FRAUD MONITOR                          [Connected]       |
+------------------------------------------------------------------+
| [156 Txns]  [6 Alertas]  [4.2% Fraude]  [$16.2M En Riesgo]     |
+------------------------------------------------------------------+
|                                                                    |
|  Tendencia Alertas 24h          |  Alertas Activas               |
|  ----___                        |  [CRITICAL] A003 - C002 - 5M   |
|        \___----___/             |  [CRITICAL] A004 - C006 - 8M   |
|                                 |  [HIGH] A001 - C002 - 5M       |
|                                 |  [HIGH] A005 - C003 - 200K     |
+------------------------------------------------------------------+
| Filtros: [Fecha] [Tipo] [Merchant] [Monto min]                   |
+------------------------------------------------------------------+
```

## Seccion 7: Verificacion

Validemos que todo el pipeline funciona correctamente.

In [ ]:
-- Snowflake SQL
-- =============================================================================
-- SECCION 7: VERIFICACION DEL PIPELINE
-- =============================================================================

-- 7.1 Verificar Dynamic Tables — datos fluyendo
SELECT 'TRANSACTIONS_ENRICHED' AS tabla, COUNT(*) AS registros 
FROM RAPPIPAY_DB.CURATED.TRANSACTIONS_ENRICHED
UNION ALL
SELECT 'FRAUD_METRICS_HOURLY', COUNT(*) 
FROM RAPPIPAY_DB.ANALYTICS.FRAUD_METRICS_HOURLY;

-- 7.2 Verificar vista enriquecida con AI
SELECT 
    alert_id, 
    severity, 
    ai_fraud_type,
    ROUND(communication_sentiment, 2) AS sentiment,
    amount,
    currency
FROM RAPPIPAY_DB.ANALYTICS.FRAUD_ALERTS_ENRICHED
ORDER BY risk_score DESC;

-- 7.3 Test rapido de Cortex AI en dato individual
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'mistral-large2',
    'En una frase, que riesgo representa esta transaccion: Transferencia de 8M COP a crypto exchange desde IP en Tokio, usuario registrado en Barranquilla Colombia, dispositivo nuevo, hora 3am local.'
) AS ai_risk_assessment;

-- 7.4 Verificar Semantic View existe y tiene metadata
SHOW SEMANTIC VIEWS IN SCHEMA RAPPIPAY_DB.ANALYTICS;

## Seccion 8: Limpieza

Cuando termines el laboratorio, ejecuta lo siguiente para eliminar todos los objetos creados.

> **Nota**: Solo ejecuta esta celda si deseas eliminar todo. Los datos y objetos no son recuperables.

In [ ]:
-- Snowflake SQL
-- =============================================================================
-- SECCION 8: LIMPIEZA (EJECUTAR SOLO AL FINALIZAR)
-- =============================================================================

-- PRECAUCION: Esto elimina TODOS los objetos del laboratorio
-- Descomenta las lineas para ejecutar

-- DROP DATABASE IF EXISTS RAPPIPAY_DB;
-- DROP WAREHOUSE IF EXISTS RAPPIPAY_WH;

---

## Resumen del Laboratorio

| Seccion | Producto Snowflake | Que construimos |
|---------|-------------------|------------------|
| Setup | SQL DDL | Base de datos, esquemas, warehouse |
| Datos | SQL DML | Transacciones, clientes, comercios, alertas ficticias |
| Pipeline | Dynamic Tables | Bronze>Silver>Gold con TARGET_LAG automatico |
| AI | Cortex AI Functions | Clasificacion, extraccion, sentimiento, resumenes |
| Analytics | Semantic Views | Consultas en lenguaje natural con Cortex Analyst |
| App | Cortex Code | Dashboard React desplegado en Snowflake App Runtime |

### Proximos pasos

- Agregar mas datos historicos y probar el refresh incremental de Dynamic Tables
- Crear alertas automaticas con `CREATE ALERT` cuando `fraud_rate_pct > 10`
- Extender la React App con panel de investigacion detallado
- Conectar Cortex Analyst a la Semantic View para que analistas hagan preguntas ad-hoc
- Explorar `AI_EXTRACT` para automatizar el parsing de documentos de compliance